# Length-Stress Test — Data Prep (multi-novel)

Per-novel truncation ladder + pooled `novel` corpus for `03`. Multiple books, not one — a single doc only gave 21 usable questions.


In [1]:
import sys
from collections import defaultdict
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import pandas as pd
from datasets import load_dataset

from src.pipeline.chuncking import plain_text_to_paragraphs

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Index questions by document (without touching document text)

Avoids reading the full-text column across ~32,700 rows.


In [2]:
DOC_KIND = "gutenberg"

ds = load_dataset("deepmind/narrativeqa", split="train")
flat = ds.flatten()

questions_by_doc: dict[str, list[dict]] = defaultdict(list)
first_row_index: dict[str, int] = {}

for i, (doc_id, kind, question, answers) in enumerate(
    zip(flat["document.id"], flat["document.kind"], flat["question.text"], flat["answers"])
):
    if kind != DOC_KIND:
        continue
    first_row_index.setdefault(doc_id, i)
    questions_by_doc[doc_id].append(
        {"question": question, "answer_variants": [a["text"] for a in answers]}
    )

print(f"{DOC_KIND} documents: {len(questions_by_doc)}")
print(f"raw questions across them: {sum(len(v) for v in questions_by_doc.values())}")

gutenberg documents: 548
raw questions across them: 16340


## 2. Pick documents by verbatim-findable answer count

Only answers with a trustworthy position (verbatim in the book) are usable here.


In [3]:
TARGET_QUESTION_COUNT = 100
MIN_QUESTIONS_PER_DOC = 5
MAX_DOCS_TO_SCAN = 60
TARGET_WORD_COUNTS = [1000, 2000, 3000, 6000, 10000, 20000, 30000, 50000]
EVAL_SLICE_WORDS = 10_000   # slice used for the pooled novel corpus (§5)
# Deliberately the eval slice, NOT max(TARGET_WORD_COUNTS): requiring every book
# to fill the 50,000-word rung would shrink the selection to 4 books and change
# the question set. Books simply skip rungs longer than they are (§3).
MIN_DOC_WORDS = EVAL_SLICE_WORDS


def strip_gutenberg_boilerplate(raw_text: str) -> str:
    """Content between the START/END markers; the license text and transcriber
    notes outside them are not part of the narrative and would skew every
    answer position computed against the document."""
    start_idx = raw_text.find("*** START")
    end_idx = raw_text.find("*** END")
    if start_idx == -1 or end_idx == -1:
        return raw_text.strip()
    return raw_text[raw_text.find("\n", start_idx) : end_idx].strip()


def word_index(text: str, char_pos: int) -> int:
    """1-indexed word position: word count before char_pos, plus 1."""
    return len(text[:char_pos].split()) + 1


def find_verbatim_answer(answer_variants: list[str], text: str) -> tuple[str, int] | None:
    """First answer variant that appears verbatim in text, with its char position."""
    for candidate in answer_variants:
        pos = text.find(candidate)
        if pos >= 0:
            return candidate, pos
    return None


ranked = sorted(questions_by_doc.items(), key=lambda kv: len(kv[1]), reverse=True)

selected_docs: list[dict] = []
kept_total = 0

for doc_id, rows in ranked[:MAX_DOCS_TO_SCAN]:
    text = strip_gutenberg_boilerplate(ds[first_row_index[doc_id]]["document"]["text"])
    if len(text.split()) < MIN_DOC_WORDS:
        continue

    kept = []
    for row in rows:
        found = find_verbatim_answer(row["answer_variants"], text)
        if found is None:
            continue
        answer_text, char_pos = found
        kept.append(
            {
                "question": row["question"],
                "answer": answer_text,
                "evidence_char_pos": char_pos,
                "evidence_word_pos": word_index(text, char_pos),
            }
        )

    if len(kept) < MIN_QUESTIONS_PER_DOC:
        continue

    selected_docs.append({"doc_id": doc_id, "text": text, "questions": kept})
    kept_total += len(kept)
    print(f"  {doc_id[:8]}  {len(text.split()):>7,} words  {len(kept):>3}/{len(rows):>3} verbatim-findable  (total {kept_total})")

    if kept_total >= TARGET_QUESTION_COUNT:
        break

print(f"\nselected {len(selected_docs)} documents, {kept_total} questions")

  00950a36   12,145 words   16/ 50 verbatim-findable  (total 16)
  3e966645   46,899 words   24/ 50 verbatim-findable  (total 40)
  6a659de3  112,273 words   19/ 50 verbatim-findable  (total 59)
  9561a72a   54,195 words   14/ 50 verbatim-findable  (total 73)
  bf55d1c1   58,233 words   14/ 50 verbatim-findable  (total 87)
  ca4b98f5  209,307 words   20/ 50 verbatim-findable  (total 107)

selected 6 documents, 107 questions


## 3. Build a truncation ladder per document

Cuts at the last paragraph boundary under the target word count.


In [4]:
def truncate_to_word_count(paragraphs: list, full_text: str, target_words: int) -> str:
    """Cut at the nearest paragraph boundary without exceeding target_words."""
    cumulative_words = 0
    cut_char_pos = 0
    for i, p in enumerate(paragraphs):
        p_words = len(p.text.split())
        if cumulative_words > 0 and cumulative_words + p_words > target_words:
            break
        cumulative_words += p_words
        next_offset = paragraphs[i + 1].char_offset if i + 1 < len(paragraphs) else len(full_text)
        cut_char_pos = next_offset
    return full_text[:cut_char_pos]


for doc in selected_docs:
    paragraphs = plain_text_to_paragraphs(doc["text"])
    doc_words = len(doc["text"].split())
    # A book shorter than a rung would make that rung a duplicate of the one
    # below it — a silently flat segment in the curve. Skip instead.
    rungs = [t for t in TARGET_WORD_COUNTS if doc_words >= t]
    doc["versions"] = {t: truncate_to_word_count(paragraphs, doc["text"], t) for t in rungs}
    doc["actual_words"] = {t: len(v.split()) for t, v in doc["versions"].items()}
    print(f"{doc['doc_id'][:8]}  {doc_words:>7,}w  paragraphs={len(paragraphs):>5}  rungs=" + ",".join(
        f"{t // 1000}k" for t in rungs
    ))

00950a36   12,145w  paragraphs=  305  rungs=1k,2k,3k,6k,10k
3e966645   46,899w  paragraphs= 1100  rungs=1k,2k,3k,6k,10k,20k,30k
6a659de3  112,273w  paragraphs= 1402  rungs=1k,2k,3k,6k,10k,20k,30k,50k
9561a72a   54,195w  paragraphs= 1961  rungs=1k,2k,3k,6k,10k,20k,30k,50k
bf55d1c1   58,233w  paragraphs=  962  rungs=1k,2k,3k,6k,10k,20k,30k,50k
ca4b98f5  209,307w  paragraphs= 3859  rungs=1k,2k,3k,6k,10k,20k,30k,50k


## 4. Save the ladder + per-version `answerable` flags

`answerable` = evidence falls within that rung's actual word count.


In [5]:
OUT_DIR = Path("data/processed/narrativeqa_length_stress")
DOCS_DIR = OUT_DIR / "docs"
SAMPLE_DIR = Path("data/sample")
DOCS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

question_rows = []
for doc in selected_docs:
    slug = doc["doc_id"][:8]
    doc_dir = DOCS_DIR / slug
    doc_dir.mkdir(exist_ok=True)
    (doc_dir / "full.txt").write_text(doc["text"], encoding="utf-8")
    for target, version_text in doc["versions"].items():
        (doc_dir / f"length_stress_v{target}.txt").write_text(version_text, encoding="utf-8")

    for local_id, q in enumerate(doc["questions"]):
        row = {
            "question_id": f"{slug}_{local_id}",
            "doc_id": doc["doc_id"],
            "doc_slug": slug,
            "question": q["question"],
            "answer": q["answer"],
            "evidence_char_pos": q["evidence_char_pos"],
            "evidence_word_pos": q["evidence_word_pos"],
            "type": "narrativeqa_extractive_subset",
        }
        for target in TARGET_WORD_COUNTS:
            # False when the book has no such rung — it cannot be shown there.
            row[f"answerable_v{target}"] = (
                target in doc["actual_words"]
                and q["evidence_word_pos"] <= doc["actual_words"][target]
            )
        question_rows.append(row)

questions_df = pd.DataFrame(question_rows)
QUESTIONS_PATH = SAMPLE_DIR / "narrativeqa_length_stress.csv"
questions_df.to_csv(QUESTIONS_PATH, index=False, encoding="utf-8-sig")

print(f"saved {len(selected_docs)} document ladders under {DOCS_DIR}")
print(f"saved: {QUESTIONS_PATH} ({len(questions_df)} questions)")
questions_df.head(3)

saved 6 document ladders under data/processed/narrativeqa_length_stress/docs
saved: data/sample/narrativeqa_length_stress.csv (107 questions)


,question_id,doc_id,doc_slug,question,answer,evidence_char_pos,evidence_word_pos,type,answerable_v1000,answerable_v2000,answerable_v3000,answerable_v6000,answerable_v10000,answerable_v20000,answerable_v30000,answerable_v50000
0,00950a36_0,00950a3641e6a28b04a6fabf6334140e2deaa9fd,00950a36,What was the name of Olivia's former master th...,Shah Amurath,3263,566,narrativeqa_extractive_subset,True,True,True,True,True,False,False,False
1,00950a36_1,00950a3641e6a28b04a6fabf6334140e2deaa9fd,00950a36,Who frees Conan from the pirates who knocked h...,Olivia,1414,233,narrativeqa_extractive_subset,True,True,True,True,True,False,False,False
2,00950a36_2,00950a3641e6a28b04a6fabf6334140e2deaa9fd,00950a36,What is the name of the city Olivia is escapin...,Akif,2333,404,narrativeqa_extractive_subset,True,True,True,True,True,False,False,False


## 5. Pooled `novel`-genre corpus for `03_qa_baseline_3conditions.ipynb`

Each doc's v10000 slice, joined with `\n\n***\n\n`.


In [6]:
SCENE_BREAK_SEPARATOR = "\n\n***\n\n"

parts = []
eval_rows = []
cursor = 0

for doc in selected_docs:
    slug = doc["doc_id"][:8]
    slice_text = doc["versions"][EVAL_SLICE_WORDS]
    cutoff_words = doc["actual_words"][EVAL_SLICE_WORDS]

    for local_id, q in enumerate(doc["questions"]):
        if q["evidence_word_pos"] > cutoff_words:
            continue
        eval_rows.append(
            {
                "question_id": f"{slug}_{local_id}",
                "doc_slug": slug,
                "question": q["question"],
                "answer": q["answer"],
                "corpus_char_pos": cursor + q["evidence_char_pos"],
                "type": "narrativeqa_extractive_subset",
            }
        )

    parts.append(slice_text)
    cursor += len(slice_text) + len(SCENE_BREAK_SEPARATOR)

novel_corpus = SCENE_BREAK_SEPARATOR.join(parts)

eval_df = pd.DataFrame(eval_rows)
eval_df["evidence_word_pos"] = [word_index(novel_corpus, p) for p in eval_df["corpus_char_pos"]]
eval_df = eval_df.rename(columns={"corpus_char_pos": "evidence_char_pos"})

# Sanity check: every remapped position must still land on its own answer text.
mismatched = [
    r["question_id"]
    for _, r in eval_df.iterrows()
    if not novel_corpus.startswith(r["answer"], r["evidence_char_pos"])
]
print("position remap mismatches:", mismatched if mismatched else "none")

CORPUS_PATH = OUT_DIR / "novel_eval_corpus.txt"
EVAL_QUESTIONS_PATH = SAMPLE_DIR / "narrativeqa_novel_eval_questions.csv"
CORPUS_PATH.write_text(novel_corpus, encoding="utf-8")
eval_df.to_csv(EVAL_QUESTIONS_PATH, index=False, encoding="utf-8-sig")

print(f"\nsaved: {CORPUS_PATH} ({len(novel_corpus.split()):,} words, {len(parts)} books)")
print(f"saved: {EVAL_QUESTIONS_PATH} ({len(eval_df)} questions)")

position remap mismatches: none

saved: data/processed/narrativeqa_length_stress/novel_eval_corpus.txt (59,027 words, 6 books)
saved: data/sample/narrativeqa_novel_eval_questions.csv (79 questions)


## 6. Cohort size per rung

Compare against the single-document version's 11/13/14/14/16.


In [7]:
summary = pd.DataFrame(
    {
        "target_words": TARGET_WORD_COUNTS,
        "answerable_questions": [questions_df[f"answerable_v{t}"].sum() for t in TARGET_WORD_COUNTS],
        "documents": [
            questions_df.loc[questions_df[f"answerable_v{t}"], "doc_slug"].nunique()
            for t in TARGET_WORD_COUNTS
        ],
        "books_with_rung": [sum(t in d["actual_words"] for d in selected_docs) for t in TARGET_WORD_COUNTS],
    }
)
summary["answerable_ratio"] = (summary["answerable_questions"] / len(questions_df)).round(2)
print(summary.to_string(index=False))
print(f"\ntotal questions: {len(questions_df)} across {len(selected_docs)} documents")
print(f"novel-genre eval set: {len(eval_df)} questions, {len(novel_corpus.split()):,} words")

 target_words  answerable_questions  documents  books_with_rung  answerable_ratio
         1000                    28          6                6              0.26
         2000                    41          6                6              0.38
         3000                    50          6                6              0.47
         6000                    56          6                6              0.52
        10000                    79          6                6              0.74
        20000                    74          5                5              0.69
        30000                    85          5                5              0.79
        50000                    64          4                4              0.60

total questions: 107 across 6 documents
novel-genre eval set: 79 questions, 59,027 words


## Summary

- narrativeqa `gutenberg` docs, ≥5 verbatim-findable answers, until 100 questions reached.
- Each doc: own 1k/2k/3k/6k/10k-word ladder.
- Questions CSV carries `doc_id`/`doc_slug` + per-rung `answerable_v*` flags.
- `novel` corpus for `03`: each doc's v10000 slice stitched with `\n\n***\n\n`.
- Next (`02`): pool answerable questions per rung across docs.
